# 3. EDA univariante

Punto 4 del enunciado: distribución de **todas** las variables.

**Responsables:** Alex (numéricas) y Vahab (categóricas y ausentes).

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(str(Path.cwd().parent / 'src'))
from config import RAW_ADZUNA, RAW_TECNOEMPLEO, PROCESSED, REFERENCE

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid')

In [ ]:
# De momento trabajamos con los datos limpios de Tecnoempleo.
# Cuando esté la unión con Jooble, se cambiará por PROCESSED / 'ofertas.csv'.
ofertas = pd.read_csv(PROCESSED / 'tecnoempleo_ofertas.csv',
                      parse_dates=['fecha_publicacion', 'fecha_obtencion'])
print(ofertas.shape)

## 3.1 Variables numéricas *(Alex)*

| Variable | Qué mide | Unidades |
|---|---|---|
| `salario_min`, `salario_max` | Extremos del rango salarial anunciado | € brutos/año |
| `salario_medio` | Punto medio del rango (variable principal del estudio) | € brutos/año |
| `salario_rango` | Amplitud del rango: máximo − mínimo (se calcula aquí) | € brutos/año |
| `antiguedad_dias` | Días entre la publicación y la descarga | días |
| `n_tecnologias` | Tecnologías del estudio que pide la oferta | número |

Las variables de salario solo existen en las ofertas que lo publican (el 30 %). Por qué faltan en el resto se estudia en el apartado 3.3.

In [ ]:
# Estilo común de los gráficos: un solo color para una sola serie,
# rejilla discreta y sin bordes superiores ni derechos.
AZUL, NARANJA, GRIS = '#2a78d6', '#eb6834', '#52514e'
plt.rcParams.update({
    'figure.figsize': (9, 4), 'figure.dpi': 110,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': GRIS, 'axes.titleweight': 'bold',
    'axes.grid': True, 'grid.color': '#e6e5e0', 'grid.linewidth': 0.8,
    'xtick.color': GRIS, 'ytick.color': GRIS, 'font.size': 10,
})
eur = plt.FuncFormatter(lambda x, _: f'{x/1000:,.0f}k €'.replace(',', '.'))

In [ ]:
ofertas['salario_rango'] = ofertas['salario_max'] - ofertas['salario_min']
con_salario = ofertas[ofertas['tiene_salario']].copy()
print(f"Ofertas con salario: {len(con_salario)} de {len(ofertas)} ({len(con_salario)/len(ofertas):.0%})")

### 3.1.1 Resumen estadístico

In [ ]:
salarios = ['salario_min', 'salario_max', 'salario_medio', 'salario_rango']
resumen = con_salario[salarios].describe(percentiles=[.05, .25, .5, .75, .95]).T
resumen['asimetría'] = con_salario[salarios].skew()
resumen['curtosis'] = con_salario[salarios].kurt()
resumen.round({**{c: 0 for c in resumen.columns}, 'asimetría': 2, 'curtosis': 2})

### 3.1.2 Distribución del salario medio

Al revisar los datos vimos que casi todos los salarios son múltiplos de 3.000 €: el portal hace elegir el salario en tramos. Por eso el histograma usa intervalos de 3.000 € alineados con esos tramos; con otros intervalos aparecerían huecos y picos que no existen.

In [ ]:
print(f"Salarios mínimos múltiplos de 3.000 €: {(con_salario['salario_min'] % 3000 == 0).mean():.0%}")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True,
                               gridspec_kw={'height_ratios': [3, 1]})
bins = np.arange(0, con_salario['salario_medio'].max() + 3000, 3000)
ax1.hist(con_salario['salario_medio'], bins=bins, color=AZUL, edgecolor='white', linewidth=1.5)
mediana = con_salario['salario_medio'].median()
media = con_salario['salario_medio'].mean()
ax1.axvline(mediana, color=GRIS, linestyle='--', linewidth=1.2)
ax1.set_ylim(0, ax1.get_ylim()[1] * 1.15)
ax1.text(mediana - 800, ax1.get_ylim()[1] * 0.97, f'mediana {mediana:,.0f} €'.replace(',', '.'), color=GRIS, va='top', ha='right')
ax1.set(title='Salario medio anunciado en Tecnoempleo', ylabel='Ofertas')
ax2.boxplot(con_salario['salario_medio'], vert=False, widths=0.6, patch_artist=True,
            boxprops=dict(facecolor='#cde2fb', color=AZUL), medianprops=dict(color=AZUL, linewidth=2),
            whiskerprops=dict(color=AZUL), capprops=dict(color=AZUL),
            flierprops=dict(marker='o', markersize=5, markerfacecolor=AZUL, markeredgecolor='white'))
ax2.set(yticks=[], xlabel='€ brutos al año')
ax2.xaxis.set_major_formatter(eur)
plt.tight_layout(); plt.show()
print(f"Media: {media:,.0f} € · mediana: {mediana:,.0f} € · desviación típica: {con_salario['salario_medio'].std():,.0f} €".replace(',', '.'))

### 3.1.3 Salario mínimo, medio y máximo

Los tres en la misma escala para comparar sus distribuciones.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
datos = [con_salario[c] for c in ['salario_min', 'salario_medio', 'salario_max']]
ax.boxplot(datos, vert=False, widths=0.55, patch_artist=True,
           tick_labels=['Mínimo', 'Medio', 'Máximo'],
           boxprops=dict(facecolor='#cde2fb', color=AZUL), medianprops=dict(color=AZUL, linewidth=2),
           whiskerprops=dict(color=AZUL), capprops=dict(color=AZUL),
           flierprops=dict(marker='o', markersize=4, markerfacecolor=AZUL, markeredgecolor='white'))
ax.xaxis.set_major_formatter(eur)
ax.set(title='Rango salarial anunciado', xlabel='€ brutos al año')
plt.tight_layout(); plt.show()

### 3.1.4 Amplitud del rango salarial

In [ ]:
fig, ax = plt.subplots()
ax.hist(con_salario['salario_rango'], bins=np.arange(0, con_salario['salario_rango'].max() + 3000, 3000),
        color=AZUL, edgecolor='white', linewidth=1.5)
ax.xaxis.set_major_formatter(eur)
ax.set(title='Diferencia entre el salario máximo y el mínimo', xlabel='€ brutos al año', ylabel='Ofertas')
plt.tight_layout(); plt.show()
print(f"Ofertas con un salario fijo (mínimo = máximo): {(con_salario['salario_rango'] == 0).mean():.1%}")
print(f"Amplitud mediana: {con_salario['salario_rango'].median():,.0f} €".replace(',', '.'))

### 3.1.5 Valores atípicos

Usamos la regla del rango intercuartílico (IQR): es atípico lo que queda por debajo de Q1 − 1,5·IQR o por encima de Q3 + 1,5·IQR. Revisamos cada caso antes de decidir qué hacer.

In [ ]:
x = con_salario['salario_medio']
q1, q3 = x.quantile([.25, .75])
iqr = q3 - q1
lim_inf, lim_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
atipicos = con_salario[(x < lim_inf) | (x > lim_sup)]
print(f"Límites: {lim_inf:,.0f} € – {lim_sup:,.0f} € · atípicos: {len(atipicos)}".replace(',', '.'))
atipicos.sort_values('salario_medio')[['titulo', 'salario_texto', 'salario_medio']]

**Salarios muy bajos.** No hay atípicos por abajo según la regla IQR, pero en la limpieza marcamos 10 ofertas de posible jornada parcial (helpdesk de fin de semana, por ejemplo). Comprobamos cuánto cambian los resultados si las quitamos:

In [ ]:
sin_parcial = con_salario[~con_salario['posible_jornada_parcial']]
pd.DataFrame({
    'Todas': con_salario['salario_medio'].agg(['count', 'mean', 'median']),
    'Sin jornada parcial': sin_parcial['salario_medio'].agg(['count', 'mean', 'median']),
}).round(0)

### 3.1.6 ¿Hace falta una escala logarítmica?

Con asimetría positiva, el logaritmo a veces acerca la distribución a una normal.

In [ ]:
print(f"Asimetría del salario medio: {x.skew():.2f} · de su logaritmo: {np.log(x).skew():.2f}")

### 3.1.7 Antigüedad de las ofertas

In [ ]:
ant = ofertas['antiguedad_dias']
fig, ax = plt.subplots()
ax.hist(ant.clip(upper=365), bins=np.arange(0, 372, 7), color=AZUL, edgecolor='white', linewidth=1)
ax.set(title='Días desde la publicación (las de más de un año, agrupadas en 365)',
       xlabel='Días', ylabel='Ofertas')
plt.tight_layout(); plt.show()
print(ant.describe(percentiles=[.5, .75, .9, .95]).round(0).to_string())
print(f"Asimetría: {ant.skew():.2f} · publicadas hace más de 90 días: {(ant > 90).sum()} · más de un año: {(ant > 365).sum()}")

### 3.1.8 Número de tecnologías por oferta

In [ ]:
conteo = ofertas['n_tecnologias'].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(conteo.index.astype(str), conteo.values, color=AZUL, width=0.6)
for i, v in enumerate(conteo.values):
    ax.text(i, v, f'{v}', ha='center', va='bottom', color=GRIS)
ax.set(title='Tecnologías del estudio que pide cada oferta', xlabel='Número de tecnologías', ylabel='Ofertas')
ax.grid(axis='x', visible=False)
plt.tight_layout(); plt.show()
print(f"Ofertas sin ninguna de las 33 tecnologías: {(ofertas['n_tecnologias'] == 0).mean():.0%}")

### 3.1.9 Conclusiones de las variables numéricas

- **Solo el 30 % de las ofertas publica salario** (392 de 1.297). Todo lo que sigue se refiere a ese subconjunto.
- **El salario medio anunciado está centrado en torno a 37.500 €** (mediana). La mitad central de las ofertas está entre 30.000 y 46.500 € brutos al año, y el 90 % entre 19.500 y 66.000 €.
- **La distribución es asimétrica hacia la derecha** (asimetría 0,90): hay pocas ofertas con salarios muy altos que tiran de la media (38.879 €) por encima de la mediana. Por eso, para comparar grupos usaremos la **mediana** y el rango intercuartílico, que no dependen de los extremos.
- **Los salarios se anuncian en tramos de 3.000 €** (el 95 % de los mínimos son múltiplos de 3.000), lo que explica la forma escalonada del histograma.
- **El rango salarial suele ser estrecho:** la diferencia mediana entre máximo y mínimo es de 6.000 €, y un 4,8 % de las ofertas da un salario fijo.
- **Valores atípicos:** 13 ofertas superan el límite superior de 71.250 €. Son todas puestos senior o directivos (arquitectos, *leads*, CTO, Head of AI) y un perfil freelance cobrado por horas. Son ofertas reales, así que **se mantienen**. Por abajo no hay atípicos según la regla IQR; las 9 ofertas con salario marcadas como posible jornada parcial apenas mueven la media (de 38.879 a 39.401 €) y no cambian la mediana, así que también se mantienen, marcadas.
- **No hace falta escala logarítmica:** el logaritmo pasa la asimetría de 0,90 a −0,42, es decir, la corrige de más. Trabajamos en euros y con estadísticos robustos.
- **Antigüedad:** la mitad de las ofertas se publicó hace 8 días o menos y el 75 % hace menos de 27, pero 96 llevan más de un año publicadas (hasta 9 años). Esta cola tan larga (asimetría 5,06) indica que el portal mantiene ofertas antiguas; en el análisis de relaciones conviene comparar resultados con y sin las ofertas no recientes.
- **Tecnologías:** el 47 % de las ofertas no pide ninguna de las 33 tecnologías del estudio, y la mayoría del resto pide una o dos. El análisis por tecnología se basará en las 693 ofertas que sí piden alguna.

## 3.2 Variables categóricas

Frecuencias de tecnología, ciudad, modalidad, fuente y tipo de contrato.

In [ ]:
# TODO (Vahab)

## 3.3 Valores ausentes

¿Qué porcentaje de ofertas publica salario? ¿Se concentra la ausencia en algún grupo?

In [ ]:
# TODO (Vahab)

## 3.4 Variables temporales

In [ ]:
# TODO (Vahab)